# GBM8360E – Lab 5, Übung 3
Lade diese 5 Dateien hoch (in Colab: Ordner-Symbol links → Hochladen), **umbenannt** wie folgt:

| Original | Neuer Name |
|---|---|
| `output\fieldmap.nii.gz` | `fieldmap.nii.gz` |
| `output\mask_box.nii.gz` | `mask_box.nii.gz` |
| `output\shim_box\fieldmap_calculated_shim_masked.nii.gz` | `shim_box.nii.gz` |
| `output\shim_nomask\fieldmap_calculated_shim_masked.nii.gz` | `shim_nomask.nii.gz` |
| `output\shim_thr\fieldmap_calculated_shim_masked.nii.gz` | `shim_thr.nii.gz` |

Dann alle Zellen ausführen (Laufzeit → Alle ausführen).

In [ ]:
!pip install -q nibabel

In [ ]:
import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to
import matplotlib.pyplot as plt

CASES = {
    "Unshimmed":      "fieldmap.nii.gz",
    "Tutorial (box)": "shim_box.nii.gz",
    "No mask":        "shim_nomask.nii.gz",
    "Threshold mask": "shim_thr.nii.gz",
}

def load_3d(path):
    img = nib.load(path)
    data = img.get_fdata()
    if data.ndim == 4:
        if data.shape[3] != 1:
            raise ValueError(f"{path} is 4D with {data.shape[3]} volumes")
        data = data[..., 0]
        img = nib.Nifti1Image(data, img.affine, img.header)
    return img, data

# Box mask (target grid) -> fieldmap grid, nearest neighbour
fmap_img, _ = load_3d(CASES["Unshimmed"])
mask_img, _ = load_3d("mask_box.nii.gz")
mask = resample_from_to(mask_img, fmap_img, order=0).get_fdata() > 0.5
print(f"Box mask: {int(mask_img.get_fdata().sum())} voxels (target grid), {int(mask.sum())} voxels (fieldmap grid)\n")

data = {n: load_3d(p)[1] for n, p in CASES.items()}
values = {}
print(f"{'Case':<16}{'Mean (Hz)':>12}{'SD (Hz)':>12}{'N voxels':>10}{'Exact 0s':>10}")
for n, d in data.items():
    assert d.shape == mask.shape, f"{n}: different grid"
    v = d[mask]
    values[n] = v
    print(f"{n:<16}{v.mean():>12.2f}{v.std():>12.2f}{v.size:>10d}{int((v == 0).sum()):>10d}")

In [ ]:
# Per-slice mean / SD inside the box (Hz) - useful for 3C Q2
print(f"{'Slice':<6}" + "".join(f"{n:>26}" for n in CASES))
for z in np.where(mask.any(axis=(0, 1)))[0]:
    line = f"{z:<6}"
    for n, d in data.items():
        v = d[..., z][mask[..., z]]
        line += f"{v.mean():>13.1f}{v.std():>13.1f}"
    print(line)

In [ ]:
# Violin plots (3C)
names = list(values)
fig, ax = plt.subplots(figsize=(8, 5))
ax.violinplot([values[n] for n in names], showmeans=True)
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.set_xticks(range(1, len(names) + 1))
ax.set_xticklabels(names)
ax.set_ylabel("B0 field offset (Hz)")
ax.set_title("Field values inside the tutorial box mask")
fig.tight_layout()
fig.savefig("fig_3C_violins.png", dpi=200)
plt.show()